In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

# Initialize Spark Session with the Kafka streaming connector package
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# 1. Load Static User Data from your CSV file
users_df = spark.read.csv("data/user_table.csv", header=True, inferSchema=True)

# 2. Read Streaming Data from Kafka
tx_schema = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", DoubleType(), True)
])

kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection") \
    .load()

# 3. Parse JSON Data
parsed_stream = kafka_stream \
    .select(from_json(col("value").cast("string"), tx_schema).alias("tx")) \
    .select("tx.*")

# 4. Fraud Logic: Amount > 10,000
fraud_stream = parsed_stream.filter(col("amount") > 10000.0)

# 5. Enrich Stream with User Details (Stream-Static Join)
enriched_fraud = fraud_stream.join(users_df, "userId")

# 6. Format for output Kafka topic
output_stream = enriched_fraud \
    .withColumn("value", to_json(struct("*")).cast("string")) \
    .select("value")

# 7. Write Stream to 'fraud-notification' Topic
query = output_stream.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "fraud-notification") \
    .option("checkpointLocation", "/tmp/checkpoints") \
    .start()

print("Streaming pipeline successfully started! Processing live transactions...")
query.awaitTermination()


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2264733f-a2b5-4669-a0c4-1e382809af99;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1192ms :: artifacts dl 39ms
	:: modules in u

Streaming pipeline successfully started! Processing live transactions...


26/06/19 06:07:33 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/19 06:07:34 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                